In [1]:
# ============================================================
# STEP 4 - CELL 1
# SETUP
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

import torch
import gc
import json
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/step4")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 4 - MULTILINGUAL TEXT EMBEDDINGS")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU     :",
        torch.cuda.get_device_name(0)
    )

    DEVICE = "cuda"

else:

    DEVICE = "cpu"

print("Device  :", DEVICE)
print("Output  :", OUTPUT_DIR)

STEP 4 - MULTILINGUAL TEXT EMBEDDINGS
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
Device  : cuda
Output  : /kaggle/working/step4


# Kiểm tra Sentence Transformers

In [2]:
# ============================================================
# STEP 4 - CELL 2
# CHECK SENTENCE TRANSFORMERS
# ============================================================

try:

    import sentence_transformers

    from sentence_transformers import (
        SentenceTransformer
    )

    print(
        "sentence-transformers:",
        sentence_transformers.__version__
    )

except ImportError:

    !pip install -q sentence-transformers

    from sentence_transformers import (
        SentenceTransformer
    )

print("✅ sentence-transformers ready")

sentence-transformers: 5.4.1
✅ sentence-transformers ready


# Load Semantic Data

In [3]:
# ============================================================
# STEP 4 - CELL 3
# DISCOVER & LOAD SEMANTIC DATA
# ============================================================

def find_file(filename):

    matches = list(
        INPUT_DIR.rglob(filename)
    )

    if not matches:

        raise FileNotFoundError(
            f"{filename} not found"
        )

    return matches[0]


review_path = find_file(
    "semantic_reviews.csv"
)

product_path = find_file(
    "semantic_products.csv"
)


print("Review file :", review_path)
print("Product file:", product_path)


semantic_reviews = pd.read_csv(
    review_path,
    low_memory=False
)

semantic_products = pd.read_csv(
    product_path,
    low_memory=False
)


print("\n" + "=" * 70)
print("DATA LOADED")
print("=" * 70)

print(
    "Semantic reviews :",
    f"{len(semantic_reviews):,}"
)

print(
    "Semantic products:",
    f"{len(semantic_products):,}"
)

print("\nReview columns:")
print(
    semantic_reviews.columns.tolist()
)

print("\nProduct columns:")
print(
    semantic_products.columns.tolist()
)

Review file : /kaggle/input/datasets/trnquangtriu/4dataset/semantic_reviews.csv
Product file: /kaggle/input/datasets/trnquangtriu/4dataset/semantic_products.csv

DATA LOADED
Semantic reviews : 714,798
Semantic products: 164,926

Review columns:
['review_id', 'user_id', 'product_id', 'rating', 'review_title', 'review_text', 'timestamp', 'verified_purchase', 'helpful_vote', 'language', 'source', 'clean_review', 'char_count', 'word_count', 'semantic_usable', 'product_name', 'category', 'brand', 'rag_document']

Product columns:
['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'currency', 'average_rating', 'review_count', 'sold_count', 'description', 'image_url', 'product_url', 'source', 'language', 'clean_product_name', 'clean_category', 'clean_subcategory', 'clean_brand', 'clean_description', 'product_text', 'product_text_word_count', 'semantic_usable']


# Audit text dùng cho embedding

In [4]:
# ============================================================
# STEP 4 - CELL 4
# EMBEDDING TEXT AUDIT
# ============================================================

print("=" * 70)
print("EMBEDDING TEXT AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# Review text
# ------------------------------------------------------------

review_texts = (
    semantic_reviews["clean_review"]
    .fillna("")
    .astype(str)
)

review_words = (
    review_texts
    .str.split()
    .str.len()
)


print("\n[1] REVIEW TEXT")
print("-" * 70)

print(
    review_words.describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)


print(
    "\nEmpty:",
    review_texts.str.strip().eq("").sum()
)


# ------------------------------------------------------------
# Product text
# ------------------------------------------------------------

product_texts = (
    semantic_products["product_text"]
    .fillna("")
    .astype(str)
)

product_words = (
    product_texts
    .str.split()
    .str.len()
)


print("\n[2] PRODUCT TEXT")
print("-" * 70)

print(
    product_words.describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)


print(
    "\nEmpty:",
    product_texts.str.strip().eq("").sum()
)


# ------------------------------------------------------------
# Source statistics
# ------------------------------------------------------------

temp_review = pd.DataFrame({
    "source":
        semantic_reviews["source"],
    "words":
        review_words
})

print("\n[3] REVIEW LENGTH BY SOURCE")

display(
    temp_review
    .groupby("source")["words"]
    .describe(
        percentiles=[
            .50,
            .90,
            .95,
            .99
        ]
    )
    .round(2)
)


temp_product = pd.DataFrame({
    "source":
        semantic_products["source"],
    "words":
        product_words
})

print("\n[4] PRODUCT LENGTH BY SOURCE")

display(
    temp_product
    .groupby("source")["words"]
    .describe(
        percentiles=[
            .50,
            .90,
            .95,
            .99
        ]
    )
    .round(2)
)

EMBEDDING TEXT AUDIT

[1] REVIEW TEXT
----------------------------------------------------------------------
count    714798.000000
mean         51.983865
std          82.632078
min           3.000000
50%          28.000000
75%          59.000000
90%         116.000000
95%         176.000000
99%         389.000000
max        3960.000000
Name: clean_review, dtype: float64

Empty: 0

[2] PRODUCT TEXT
----------------------------------------------------------------------
count    164926.000000
mean         37.884324
std          17.733502
min           6.000000
50%          34.000000
75%          45.000000
90%          67.000000
95%          72.000000
99%          81.000000
max         342.000000
Name: product_text, dtype: float64

Empty: 0

[3] REVIEW LENGTH BY SOURCE


,count,mean,std,min,50%,90%,95%,99%,max
source,,,,,,,,,
amazon,683832.0,53.03,84.25,3.0,28.0,119.0,180.0,396.0,3960.0
shopee,30966.0,28.93,17.61,3.0,26.0,48.0,59.0,88.0,454.0



[4] PRODUCT LENGTH BY SOURCE


,count,mean,std,min,50%,90%,95%,99%,max
source,,,,,,,,,
amazon,121917.0,29.27,8.79,6.0,30.0,40.0,42.0,47.0,342.0
shopee,1434.0,25.19,5.65,11.0,25.0,32.0,34.0,39.0,53.0
tiki,41575.0,63.58,11.69,12.0,65.0,76.0,79.0,87.0,127.0


# Load Multilingual Model

In [5]:
# ============================================================
# STEP 4 - CELL 5
# LOAD MULTILINGUAL EMBEDDING MODEL
# ============================================================

MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("=" * 70)
print("LOAD EMBEDDING MODEL")
print("=" * 70)

print("Model :", MODEL_NAME)
print("Device:", DEVICE)


model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)


EMBEDDING_DIM = (
    model.get_sentence_embedding_dimension()
)

MAX_SEQ_LENGTH = (
    model.max_seq_length
)


print(
    "\nEmbedding dimension:",
    EMBEDDING_DIM
)

print(
    "Max sequence length:",
    MAX_SEQ_LENGTH
)

print("\n✅ Model loaded")

LOAD EMBEDDING MODEL
Model : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding dimension: 384
Max sequence length: 128

✅ Model loaded


# Sanity Test Anh–Việt

In [6]:
# ============================================================
# STEP 4 - CELL 6
# CROSS-LINGUAL SANITY TEST
# ============================================================

print("=" * 70)
print("CROSS-LINGUAL SANITY TEST")
print("=" * 70)

test_sentences = [

    "tai nghe bluetooth không dây",

    "wireless bluetooth headphones",

    "giày thể thao nam",

    "men's sports shoes",

    "máy ảnh kỹ thuật số",

    "digital camera"
]


test_embeddings = model.encode(

    test_sentences,

    batch_size=6,

    convert_to_numpy=True,

    normalize_embeddings=True,

    show_progress_bar=False
)


similarity = (
    test_embeddings
    @ test_embeddings.T
)


similarity_df = pd.DataFrame(

    similarity,

    index=test_sentences,

    columns=test_sentences
)


display(
    similarity_df.round(3)
)


print("\nExpected semantic pairs:")

pairs = [
    (0, 1),
    (2, 3),
    (4, 5)
]

for i, j in pairs:

    print(
        f"{test_sentences[i]}"
        f"  <->  "
        f"{test_sentences[j]}"
        f" = {similarity[i, j]:.4f}"
    )

CROSS-LINGUAL SANITY TEST


,tai nghe bluetooth không dây,wireless bluetooth headphones,giày thể thao nam,men's sports shoes,máy ảnh kỹ thuật số,digital camera
tai nghe bluetooth không dây,1.000,0.865,0.098,0.104,0.272,0.249
wireless bluetooth headphones,0.865,1.000,0.124,0.121,0.249,0.232
giày thể thao nam,0.098,0.124,1.000,0.888,0.104,0.082
men's sports shoes,0.104,0.121,0.888,1.000,0.070,0.052
máy ảnh kỹ thuật số,0.272,0.249,0.104,0.070,1.000,0.974
digital camera,0.249,0.232,0.082,0.052,0.974,1.000



Expected semantic pairs:
tai nghe bluetooth không dây  <->  wireless bluetooth headphones = 0.8648
giày thể thao nam  <->  men's sports shoes = 0.8881
máy ảnh kỹ thuật số  <->  digital camera = 0.9738


# Test batch trước khi encode toàn bộ

In [7]:
# ============================================================
# STEP 4 - CELL 7
# GPU BATCH TEST
# ============================================================

print("=" * 70)
print("GPU BATCH TEST")
print("=" * 70)

BATCH_SIZE = 256

test_size = min(
    2048,
    len(product_texts)
)

test_batch = (
    product_texts
    .iloc[:test_size]
    .tolist()
)


with torch.inference_mode():

    test_vectors = model.encode(

        test_batch,

        batch_size=BATCH_SIZE,

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=True
    )


print(
    "\nTest shape:",
    test_vectors.shape
)

print(
    "dtype:",
    test_vectors.dtype
)

print(
    "First vector norm:",
    np.linalg.norm(
        test_vectors[0]
    )
)


del test_vectors

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Batch test passed")

GPU BATCH TEST


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Test shape: (2048, 384)
dtype: float32
First vector norm: 1.0

✅ Batch test passed


# Encode toàn bộ Product

In [8]:
# ============================================================
# STEP 4 - CELL 8
# GENERATE PRODUCT EMBEDDINGS
# ============================================================

print("=" * 70)
print("GENERATE PRODUCT EMBEDDINGS")
print("=" * 70)

print(
    "Products:",
    f"{len(product_texts):,}"
)

print(
    "Batch size:",
    BATCH_SIZE
)


with torch.inference_mode():

    product_embeddings = model.encode(

        product_texts.tolist(),

        batch_size=BATCH_SIZE,

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=True
    )


product_embeddings = (
    product_embeddings
    .astype(np.float32)
)


print("\nShape:", product_embeddings.shape)
print("dtype:", product_embeddings.dtype)


product_norms = np.linalg.norm(
    product_embeddings,
    axis=1
)


print(
    "Mean vector norm:",
    product_norms.mean()
)

print(
    "Min vector norm :",
    product_norms.min()
)

print(
    "Max vector norm :",
    product_norms.max()
)


assert (
    product_embeddings.shape[0]
    == len(semantic_products)
)

assert (
    product_embeddings.shape[1]
    == EMBEDDING_DIM
)

assert np.isfinite(
    product_embeddings
).all()

print("\n✅ Product embeddings generated")

GENERATE PRODUCT EMBEDDINGS
Products: 164,926
Batch size: 256


Batches:   0%|          | 0/645 [00:00<?, ?it/s]


Shape: (164926, 384)
dtype: float32
Mean vector norm: 1.0
Min vector norm : 0.9999999
Max vector norm : 1.0000001

✅ Product embeddings generated


# Save Product Embeddings ngay

In [9]:
# ============================================================
# STEP 4 - CELL 9
# SAVE PRODUCT EMBEDDINGS
# ============================================================

product_embedding_path = (
    OUTPUT_DIR /
    "product_embeddings.npy"
)

product_metadata_path = (
    OUTPUT_DIR /
    "product_embedding_metadata.csv"
)


np.save(
    product_embedding_path,
    product_embeddings
)


product_metadata = (
    semantic_products
    .copy()
    .reset_index(drop=True)
)

product_metadata[
    "product_embedding_id"
] = np.arange(
    len(product_metadata)
)


product_metadata.to_csv(
    product_metadata_path,
    index=False
)


print(
    "Embedding:",
    product_embedding_path
)

print(
    "Metadata :",
    product_metadata_path
)

print(
    "\nVectors:",
    f"{len(product_metadata):,}"
)

print(
    "Shape:",
    product_embeddings.shape
)

print("\n✅ Product artifacts saved")

Embedding: /kaggle/working/step4/product_embeddings.npy
Metadata : /kaggle/working/step4/product_embedding_metadata.csv

Vectors: 164,926
Shape: (164926, 384)

✅ Product artifacts saved


# Chuẩn bị Review Metadata

In [10]:
# ============================================================
# STEP 4 - CELL 10
# PREPARE REVIEW EMBEDDING DATA
# ============================================================

print("=" * 70)
print("PREPARE REVIEW EMBEDDING DATA")
print("=" * 70)

review_embedding_texts = (
    semantic_reviews["clean_review"]
    .fillna("")
    .astype(str)
    .tolist()
)


review_metadata_columns = [

    "review_id",
    "user_id",
    "product_id",
    "rating",
    "timestamp",
    "language",
    "source",
    "clean_review"
]

review_metadata_columns = [

    col
    for col in review_metadata_columns

    if col in semantic_reviews.columns
]


review_metadata = (
    semantic_reviews[
        review_metadata_columns
    ]
    .copy()
    .reset_index(drop=True)
)


review_metadata[
    "review_embedding_id"
] = np.arange(
    len(review_metadata)
)


print(
    "Reviews:",
    f"{len(review_embedding_texts):,}"
)

print(
    "Metadata:",
    review_metadata.shape
)

print(
    "\nColumns:"
)

print(
    review_metadata.columns.tolist()
)

PREPARE REVIEW EMBEDDING DATA
Reviews: 714,798
Metadata: (714798, 9)

Columns:
['review_id', 'user_id', 'product_id', 'rating', 'timestamp', 'language', 'source', 'clean_review', 'review_embedding_id']


# Generate Review Embeddings

In [11]:
# ============================================================
# STEP 4 - CELL 11
# GENERATE REVIEW EMBEDDINGS
# ============================================================

print("=" * 70)
print("GENERATE REVIEW EMBEDDINGS")
print("=" * 70)

print(
    "Reviews:",
    f"{len(review_embedding_texts):,}"
)

print(
    "Batch size:",
    BATCH_SIZE
)


with torch.inference_mode():

    review_embeddings = model.encode(

        review_embedding_texts,

        batch_size=BATCH_SIZE,

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=True
    )


review_embeddings = (
    review_embeddings
    .astype(np.float32)
)


print("\nShape:", review_embeddings.shape)
print("dtype:", review_embeddings.dtype)


review_norms = np.linalg.norm(
    review_embeddings,
    axis=1
)


print(
    "Mean vector norm:",
    review_norms.mean()
)

print(
    "Min vector norm :",
    review_norms.min()
)

print(
    "Max vector norm :",
    review_norms.max()
)


assert (
    review_embeddings.shape[0]
    == len(review_metadata)
)

assert (
    review_embeddings.shape[1]
    == EMBEDDING_DIM
)

assert np.isfinite(
    review_embeddings
).all()

print("\n✅ Review embeddings generated")

GENERATE REVIEW EMBEDDINGS
Reviews: 714,798
Batch size: 256


Batches:   0%|          | 0/2793 [00:00<?, ?it/s]


Shape: (714798, 384)
dtype: float32
Mean vector norm: 1.0
Min vector norm : 0.9999998
Max vector norm : 1.0000001

✅ Review embeddings generated


# Save Review Embeddings

In [12]:
# ============================================================
# STEP 4 - CELL 12
# SAVE REVIEW EMBEDDINGS
# ============================================================

review_embedding_path = (
    OUTPUT_DIR /
    "review_embeddings.npy"
)

review_metadata_path = (
    OUTPUT_DIR /
    "review_embedding_metadata.csv"
)


np.save(
    review_embedding_path,
    review_embeddings
)


review_metadata.to_csv(
    review_metadata_path,
    index=False
)


print(
    "Embedding:",
    review_embedding_path
)

print(
    "Metadata :",
    review_metadata_path
)

print(
    "\nVectors:",
    f"{len(review_metadata):,}"
)

print(
    "Shape:",
    review_embeddings.shape
)

print("\n✅ Review artifacts saved")

Embedding: /kaggle/working/step4/review_embeddings.npy
Metadata : /kaggle/working/step4/review_embedding_metadata.csv

Vectors: 714,798
Shape: (714798, 384)

✅ Review artifacts saved


# Cross-lingual Retrieval Test trên sản phẩm thật

In [13]:
# ============================================================
# STEP 4 - CELL 13
# REAL PRODUCT RETRIEVAL SANITY TEST
# ============================================================

print("=" * 70)
print("REAL PRODUCT RETRIEVAL TEST")
print("=" * 70)


def semantic_product_search(
    query,
    top_k=5
):

    query_vector = model.encode(

        [query],

        convert_to_numpy=True,

        normalize_embeddings=True,

        show_progress_bar=False
    )[0].astype(np.float32)


    scores = (
        product_embeddings
        @ query_vector
    )


    top_indices = np.argpartition(
        -scores,
        min(
            top_k,
            len(scores)
        ) - 1
    )[:top_k]


    top_indices = (
        top_indices[
            np.argsort(
                -scores[
                    top_indices
                ]
            )
        ]
    )


    cols = [
        "product_id",
        "source",
        "product_name",
        "price",
        "currency"
    ]

    cols = [
        c
        for c in cols
        if c in product_metadata.columns
    ]


    result = (
        product_metadata
        .iloc[top_indices][cols]
        .copy()
    )

    result[
        "similarity"
    ] = scores[
        top_indices
    ]

    return result.reset_index(
        drop=True
    )

REAL PRODUCT RETRIEVAL TEST


In [14]:
queries = [

    "tai nghe bluetooth",

    "wireless bluetooth headphones",

    "giày nam",

    "men shoes",

    "camera for photography"
]


for query in queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    display(
        semantic_product_search(
            query,
            top_k=5
        )
    )


QUERY: tai nghe bluetooth


,product_id,source,product_name,price,currency,similarity
0,amazon_B06XX7QKL2,amazon,"Urbanears Stadion in-Ear Active Wireless Bluetooth Headset, Trail (04091870), Blue",32.99,USD,0.750951
1,amazon_B074PCB3XB,amazon,SYLVANIA SP328-Turquoise Portable Bluetooth Speaker,23.99,USD,0.742384
2,amazon_B083YZ61VN,amazon,JBL Duet Mini 2 Bluetooth Wireless In-Ear Headphones Blue,50.49,USD,0.740766
3,amazon_B0C5PC4J6W,amazon,"BlueParrott S450-XT Voice-Controlled Bluetooth Headset – Industry Leading Sound with Long Wireless Range, Extreme Co...",172.70,USD,0.732338
4,amazon_B0B7W59NZL,amazon,"Besign BE-RCA Pro Long Range Bluetooth Adapter for Bluetooth Music Streaming, Black",24.99,USD,0.729203



QUERY: wireless bluetooth headphones


,product_id,source,product_name,price,currency,similarity
0,amazon_B0C1ZHJRGB,amazon,Wireless Earbuds Bluetooth Headphones 48hrs Play Back Sport Earphones with LED Display Over-Ear Buds with Earhooks B...,29.99,USD,0.856290
1,amazon_B081PS7YHC,amazon,Wireless Earbuds Bluetooth Headset with Microphone Sports Headset Touch TWS Stereo Headset in-Ear Bluetooth 50H Play...,19.99,USD,0.847187
2,amazon_B0BWK6B2T4,amazon,"Sony WH-CH520 Wireless Headphones Bluetooth On-Ear Headset with Microphone, Blue New",38.00,USD,0.846779
3,amazon_B09L3X7KPH,amazon,"LOBKIN Bluetooth Headphone Over Ear, Stereo Wireless Headset with Microphone, Foldable Wireless and Wired Headphones...",37.99,USD,0.845866
4,amazon_B091TNX39Z,amazon,"Bluetooth Headphones Wireless Neckband Bluetooth 5.0 Headset with 20 H Playtime, 10 mm Drivers, Magnetic Earbuds, Cr...",15.99,USD,0.842774



QUERY: giày nam


,product_id,source,product_name,price,currency,similarity
0,tiki_138561248,tiki,Tất hài nữ đi giày búp bê,35000.0,VND,0.700324
1,tiki_171644838,tiki,Lót Giày Thể Thao Êm Chân Thoáng Khí Chống Hôi Chân Khi Sử Dụng Giày Thường Xuyên -LG03,18000.0,VND,0.693778
2,tiki_196596754,tiki,Combo Vệ Sinh Giày Đa Năng Tẩy Vết Ố 420ML Và Bàn Chải Đen,150000.0,VND,0.692189
3,tiki_101063188,tiki,Chai Xịt Bọt Tuyết Vệ Sinh Giày Dép Tẩy Trắng Sneaker Cao Cấp 206736,45000.0,VND,0.689907
4,tiki_86017517,tiki,giày nữ da bò cao gót 6 cm - xeim hàng mới thanh toán,99000.0,VND,0.681627



QUERY: men shoes


,product_id,source,product_name,price,currency,similarity
0,tiki_84936233,tiki,Giày thể thao nam sneaker đế 2 lớp nam tính G183,55000.00,VND,0.699830
1,amazon_B00131GCAC,amazon,Manfrotto 565 Rubber Tripod Shoes for Twin Spiked Feet - Set of 3 (Black),39.88,USD,0.688723
2,tiki_181488092,tiki,Giày thể thao sneaker nam nữ đế cao thời trang,175000.00,VND,0.682032
3,tiki_173252858,tiki,Giày nam trẻ trung buộc dây 20896,328000.00,VND,0.678208
4,tiki_177584897,tiki,Giày nam trẻ trung buộc dây 20896,440880.00,VND,0.678208



QUERY: camera for photography


,product_id,source,product_name,price,currency,similarity
0,amazon_B07WFNWTV4,amazon,"BEIYANG Backdrop Stand, 7.5 FT x 10 FT Adjustable Photography Background Support System Kit with Carrying Bag for Ph...",39.98,USD,0.703087
1,amazon_B01BZ76Q54,amazon,"LimoStudio 24"" x 24"" / 60cm x 60cm Black & White Acrylic Reflective Table Top Display Background Boards for Photogra...",48.90,USD,0.700620
2,amazon_B08XVNSW72,amazon,"ISHARE Digital Camera for Photography with 2.8’’LCD, 20MP HD Photography Camera Rechargeable Point and Shoot Camera ...",39.99,USD,0.693807
3,amazon_B09SQ66VZN,amazon,Billingham Hadley Pro Canvas Bag for Camera - Parent Asin,276.00,USD,0.692460
4,amazon_B083DLZT3L,amazon,Neewer 8.5x10ft Backdrop Stand Support Continuous Lighting Kit for Photography Video Shooting,219.49,USD,0.689691


# Final Embedding Audit

In [15]:
# ============================================================
# STEP 4 - CELL 14
# FINAL EMBEDDING AUDIT
# ============================================================

print("=" * 70)
print("FINAL EMBEDDING AUDIT")
print("=" * 70)


checks = {

    "product_rows":
        product_embeddings.shape[0]
        == len(product_metadata),

    "review_rows":
        review_embeddings.shape[0]
        == len(review_metadata),

    "product_dimension":
        product_embeddings.shape[1]
        == EMBEDDING_DIM,

    "review_dimension":
        review_embeddings.shape[1]
        == EMBEDDING_DIM,

    "product_finite":
        np.isfinite(
            product_embeddings
        ).all(),

    "review_finite":
        np.isfinite(
            review_embeddings
        ).all(),

    "product_ids_unique":
        product_metadata[
            "product_id"
        ].is_unique,

    "review_ids_unique":
        review_metadata[
            "review_id"
        ].is_unique,

    "product_embedding_ids":
        np.array_equal(

            product_metadata[
                "product_embedding_id"
            ].values,

            np.arange(
                len(product_metadata)
            )
        ),

    "review_embedding_ids":
        np.array_equal(

            review_metadata[
                "review_embedding_id"
            ].values,

            np.arange(
                len(review_metadata)
            )
        )
}


for name, status in checks.items():

    print(
        ("✅" if status else "❌"),
        f"{name:<30}:",
        status
    )


print("\n" + "-" * 70)

print(
    "Product embeddings:",
    product_embeddings.shape
)

print(
    "Review embeddings :",
    review_embeddings.shape
)

print(
    "Embedding dimension:",
    EMBEDDING_DIM
)


assert all(
    checks.values()
)

print(
    "\n✅ All embedding consistency checks passed"
)

FINAL EMBEDDING AUDIT
✅ product_rows                  : True
✅ review_rows                   : True
✅ product_dimension             : True
✅ review_dimension              : True
✅ product_finite                : True
✅ review_finite                 : True
✅ product_ids_unique            : True
✅ review_ids_unique             : True
✅ product_embedding_ids         : True
✅ review_embedding_ids          : True

----------------------------------------------------------------------
Product embeddings: (164926, 384)
Review embeddings : (714798, 384)
Embedding dimension: 384

✅ All embedding consistency checks passed


# Save Model

In [16]:
# ============================================================
# STEP 4 - CELL 15
# SAVE EMBEDDING CONFIG
# ============================================================

model_info = {

    "model_name":
        MODEL_NAME,

    "embedding_dimension":
        int(
            EMBEDDING_DIM
        ),

    "max_sequence_length":
        int(
            MAX_SEQ_LENGTH
        ),

    "normalized_embeddings":
        True,

    "similarity":
        "cosine_via_inner_product",

    "review_embedding_text":
        "clean_review",

    "product_embedding_text":
        "product_text",

    "review_vectors":
        int(
            len(review_metadata)
        ),

    "product_vectors":
        int(
            len(product_metadata)
        )
}


model_info_path = (
    OUTPUT_DIR /
    "embedding_config.json"
)


with open(
    model_info_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_info,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        model_info,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nSaved:",
    model_info_path
)

{
  "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_dimension": 384,
  "max_sequence_length": 128,
  "normalized_embeddings": true,
  "similarity": "cosine_via_inner_product",
  "review_embedding_text": "clean_review",
  "product_embedding_text": "product_text",
  "review_vectors": 714798,
  "product_vectors": 164926
}

Saved: /kaggle/working/step4/embedding_config.json


In [17]:
# ============================================================
# STEP 4 - CELL 16
# OUTPUT SUMMARY
# ============================================================

print("=" * 70)
print("STEP 4 OUTPUT SUMMARY")
print("=" * 70)


files = [

    OUTPUT_DIR /
    "product_embeddings.npy",

    OUTPUT_DIR /
    "product_embedding_metadata.csv",

    OUTPUT_DIR /
    "review_embeddings.npy",

    OUTPUT_DIR /
    "review_embedding_metadata.csv",

    OUTPUT_DIR /
    "embedding_config.json"
]


for path in files:

    if path.exists():

        size_mb = (
            path.stat().st_size /
            1024**2
        )

        print(
            f"{path.name:<35}"
            f"{size_mb:>10.2f} MB"
        )

    else:

        print(
            "❌ Missing:",
            path.name
        )


print(
    "\nOutput:",
    OUTPUT_DIR
)

print("\n" + "=" * 70)
print("✅ STEP 4 COMPLETED")
print("=" * 70)

STEP 4 OUTPUT SUMMARY
product_embeddings.npy                 241.59 MB
product_embedding_metadata.csv         112.27 MB
review_embeddings.npy                 1047.07 MB
review_embedding_metadata.csv          261.55 MB
embedding_config.json                    0.00 MB

Output: /kaggle/working/step4

✅ STEP 4 COMPLETED
